# Deploy — Detektor Poster AI vs Non-AI

Notebook ini menjalankan aplikasi web Gradio dari folder `Deep Learning/deploy_hf` di Drive.

- **Opsi A — demo cepat (Colab):** menghasilkan link publik sementara. Link mati kalau runtime Colab berhenti.
- **Opsi B — permanen (Hugging Face Spaces, gratis):** mengunggah aplikasi ke Hugging Face sehingga punya alamat tetap, misalnya `https://huggingface.co/spaces/<username>/detektor-poster-ai`.

Kalau notebook utama sudah dijalankan dan menghasilkan `Deep Learning/hasil_model/detektor_poster_ai.keras`, model itu yang dipakai. Kalau belum, dipakai model bawaan di folder `deploy_hf`.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Tidak berjalan di Colab, mount Drive dilewati.')

In [ ]:
import os, json, shutil

DRIVE_DIR  = '/content/drive/MyDrive/Deep Learning'
DEPLOY_SRC = f'{DRIVE_DIR}/deploy_hf'
HASIL      = f'{DRIVE_DIR}/hasil_model'
APP_DIR    = '/content/deploy'

assert os.path.exists(f'{DEPLOY_SRC}/app.py'), f'Folder {DEPLOY_SRC} tidak ditemukan atau tidak berisi app.py'
shutil.rmtree(APP_DIR, ignore_errors=True)
shutil.copytree(DEPLOY_SRC, APP_DIR, ignore=shutil.ignore_patterns('*.ipynb'))

# Pakai model hasil training terbaru kalau ada (model dan ringkasannya harus dari run yang sama)
if os.path.exists(f'{HASIL}/detektor_poster_ai.keras') and os.path.exists(f'{HASIL}/ringkasan_hasil.json'):
    for f in ['detektor_poster_ai.keras', 'ringkasan_hasil.json']:
        shutil.copy(f'{HASIL}/{f}', f'{APP_DIR}/{f}')
    sumber = 'hasil_model (hasil training notebook utama)'
else:
    sumber = 'model bawaan folder deploy_hf'

s = json.load(open(f'{APP_DIR}/ringkasan_hasil.json'))
print('Sumber model :', sumber)
print('Arsitektur   :', s['model'], '| lapisan potong:', s['lapisan_potong'])
print('Akurasi test :', f"{s['akurasi_test']:.1%}", '| akurasi out-of-fold:', f"{s['akurasi_out_of_fold']:.1%}")
print('Isi folder   :', sorted(os.listdir(APP_DIR)))

In [ ]:
!pip -q install gradio==6.32.0

## Opsi A — Demo cepat di Colab

Jalankan cell di bawah, tunggu sampai muncul baris `Running on public URL: https://....gradio.live`, lalu buka link tersebut. Cell akan terus berjalan selama aplikasi aktif; hentikan dengan tombol stop.

In [ ]:
!cd /content/deploy && python app.py --share

## Opsi B — Deploy permanen ke Hugging Face Spaces

1. Buat akun gratis di https://huggingface.co (kalau belum punya).
2. Buka **Settings → Access Tokens → Create new token**, pilih tipe **Write**, lalu salin token-nya.
3. Jalankan cell di bawah dan tempel token saat diminta.

Setelah unggahan selesai, Hugging Face akan membangun aplikasinya sekitar 3 sampai 5 menit. Status build bisa dilihat di tab **Logs** pada halaman Space.

In [ ]:
!pip -q install huggingface_hub
from huggingface_hub import login, create_repo, upload_folder

SPACE_NAME = 'detektor-poster-ai'     # boleh diganti, tanpa spasi

login()                               # tempel token Write dari Hugging Face
repo = create_repo(SPACE_NAME, repo_type='space', space_sdk='gradio', exist_ok=True)
upload_folder(folder_path=APP_DIR, repo_id=repo.repo_id, repo_type='space',
              commit_message='Deploy detektor poster AI')
print('Selesai. Buka:', f'https://huggingface.co/spaces/{repo.repo_id}')